In [ ]:
from train_and_eval import train_and_evaluate
from models import BiGRUEncoder, GRUDecoder, CNNGRUDecoder

In [ ]:
params = {  "neural_dim": 512,
            "n_units": 256,
            "n_days": 2,
            "n_classes": 41,
            "rnn_dropout": 0.4,
            "input_dropout": 0.2,
            "n_layers": 3,
            "patch_size": 14,
            "patch_stride": 4,
 }

In [3]:
preds, model, plots = train_and_evaluate(GRUDecoder, params, n_epochs=3, learning_rate=1e-2)

  2%|▏         | 1/45 [00:05<03:53,  5.30s/it]


Data loaded and merged.


100%|██████████| 80/80 [02:08<00:00,  1.61s/it]


Calculating sequence accuracy...
Epoch 1/3 | train loss: 3.8519 | val loss: 3.5032 | lev acc: 0.0625 | seq acc: 0.0000


100%|██████████| 80/80 [02:09<00:00,  1.62s/it]


Calculating sequence accuracy...
Epoch 2/3 | train loss: 3.2887 | val loss: 3.3904 | lev acc: 0.0667 | seq acc: 0.0000


100%|██████████| 80/80 [02:14<00:00,  1.68s/it]


Calculating sequence accuracy...
Epoch 3/3 | train loss: 3.1830 | val loss: 3.6297 | lev acc: 0.0770 | seq acc: 0.0000
Loaded best model from epoch 3 (val loss = 3.6297) with val lev acc = 0.0770
Generated predictions for 35 test samples


In [ ]:
import os
import torch

save_dir = "BaselineGRU"
os.makedirs(save_dir, exist_ok=True)

torch.save(
    model.state_dict(),
    os.path.join(save_dir, "gru_decoder.pth")
)

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GRUDecoder(**params)     # must match training params
state_dict = torch.load(
    "BaselineGRU/gru_decoder_best.pth",
    map_location=device
)

model.load_state_dict(state_dict)
model.to(device)
model.eval()

In [ ]:
import matplotlib.pyplot as plt

train_losses = plots["train_losses"]
val_losses = plots["val_losses"]
val_lev_accs = plots["val_lev_accs"]
val_seq_accs = plots["val_seq_accs"]

epochs = range(1, len(train_losses) + 1)

plt.figure(figsize=(14, 5))

# --- Losses ---
plt.subplot(1, 2, 1)
plt.plot(epochs, train_losses, label="Train Loss", marker="o")
plt.plot(epochs, val_losses, label="Val Loss", marker="o")
plt.xlabel("Epoch")
plt.ylabel("CTC Loss")
plt.title("Training / Validation Loss")
plt.legend()
plt.grid(True)

# --- Accuracies ---
plt.subplot(1, 2, 2)
plt.plot(epochs, val_lev_accs, label="Val Levenshtein Acc", marker="o")
plt.plot(epochs, val_seq_accs, label="Val Sequence Acc", marker="o")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy Metrics")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
preds[2]

In [ ]:
# LOGIT_TO_PHONEME = [
# 'BLANK',    # "BLANK" = CTC blank symbol
# 'AA', 'AE', 'AH', 'AO', 'AW',
# 'AY', 'B', 'CH', 'D', 'DH',
# 'EH', 'ER', 'EY', 'F', 'G',
# 'HH', 'IH', 'IY', 'JH', 'K',
# 'L', 'M', 'N', 'NG', 'OW',
# 'OY', 'P', 'R', 'S', 'SH',
# 'T', 'TH', 'UH', 'UW', 'V',
# 'W', 'Y', 'Z', 'ZH',
# ' | ',    # "|" = silence token
# ]

# def decode_logits_to_phonemes(indices, vocab):
#     """
#     CTC-style decoding: collapse repeats, remove blanks,
#     then map indices to phoneme strings.
#     """
#     decoded = []
#     prev = None

#     for idx in indices:
#         if idx != 0:  # 0 = BLANK
#             decoded.append(vocab[idx])
        

#     return decoded


In [ ]:
from utils import indexes_to_phonemes
indexes_to_phonemes(preds[2])